# 9) Rolling horizon: operating with limited foresight

## Objectives

At the end of this notebook you should be able to:

- explain why real operation is planned in **rolling windows** rather than for a whole week or year at once
- implement a rolling horizon yourself — plan a window, **commit** its first part, carry the storage level forward, move on — and with PyPSA's `optimize_with_rolling_horizon`
- show the **end-of-horizon problem**: a short window sees no value in water left over and empties the reservoir
- fix it with a **look-ahead** or with a **terminal water value** — and show that the right water value closes the gap to perfect foresight completely
- compare rolling strategies fairly, and avoid two silent traps in PyPSA's rolling horizon

Every model so far saw the whole period at once: lesson 7's reservoir knew on Monday what Friday evening would bring. Real operators do not. A day-ahead schedule is made each day for the next day, with the reservoir level the previous day left behind. This lesson runs lesson 7's week that way. All numbers are **illustrative**.

In [1]:
import logging
import warnings

import linopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### Lesson 7's week, with a starting level

In [2]:
T = 168
h = np.arange(T)
hour, day = h % 24, h // 24
rng = np.random.default_rng(3)
load = (2000 + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0) + 750 * np.exp(-((hour - 19) / 2.0) ** 2)
        - 150 * (day >= 5) + rng.normal(0, 30, T))
cloud = np.repeat([0.9, 0.6, 1.0, 0.8, 0.4, 1.0, 0.9], 24)
free = (2500 * 0.75 * np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * cloud
        + 1000 * np.clip(0.3 + 0.25 * np.sin(2 * np.pi * h / 96), 0.02, 0.9) + 300.0)
thermal = pd.DataFrame({"p_nom": [350, 900, 600, 800, 400], "mc": [12.0, 89.1, 94.2, 104.1, 128.9]},
                       index=pd.Index(["nuclear", "ccgt", "coal", "imports", "ocgt"], name="gen"))
INFLOW, E_MAX, TURBINE = 120.0, 30_000.0, 1000.0
E_START = 3000.0                          # MWh in the reservoir on Monday at 00:00
WATER_VALUE = 94.2                        # lesson 7's water value for this week (EUR/MWh)

### One window

The building block is lesson 7's model for an arbitrary window of hours, with three changes. It starts from a given level instead of being cyclic. It may require a minimum level at the end. And it may give the water left at the end a **terminal value** — a reward of v €/MWh for every MWh still in the reservoir when the window closes:

minimise Σ marginal cost · g − v · e[last hour]

In [3]:
def window(hours, e_start, terminal_value=0.0, end_min=0.0):
    """Lesson 7's dispatch LP for the given hours. Returns generation (DataFrame), turbine (MW), level (MWh)."""
    n_t = len(hours)
    m = linopy.Model()
    t, G = pd.RangeIndex(n_t, name="t"), thermal.index
    g = m.add_variables(lower=0, upper=xr.DataArray(np.tile(thermal.p_nom.to_numpy(), (n_t, 1)), coords=[t, G]), name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(free[hours], coords=[t]), name="free")
    d = m.add_variables(lower=0, upper=TURBINE, coords=[t], name="turbine")
    s = m.add_variables(lower=0, coords=[t], name="spill")
    e = m.add_variables(lower=0, upper=E_MAX, coords=[t], name="level")
    later = xr.DataArray(np.arange(n_t) > 0, coords=[t])
    m.add_constraints(e - e.shift(t=1) + d + s == INFLOW, name="water", mask=later)
    m.add_constraints(e.sel(t=0) + d.sel(t=0) + s.sel(t=0) == INFLOW + e_start, name="water_first_hour")
    if end_min > 0:
        m.add_constraints(e.sel(t=n_t - 1) >= end_min, name="end_level")
    m.add_constraints(g.sum("gen") + f + d == xr.DataArray(load[hours], coords=[t]), name="balance")
    m.add_objective((g * xr.DataArray(thermal.mc.to_numpy(), coords=[G])).sum() - terminal_value * e.sel(t=n_t - 1))
    m.solve(solver_name="highs", output_flag=False)
    return g.solution.to_pandas(), d.solution.to_numpy(), e.solution.to_numpy()


def fuel_cost(gen):
    return float((gen * thermal.mc.to_numpy()).sum().sum())


def adjusted_cost(cost, end_level):
    """Fuel cost plus the value of water used beyond the starting level (minus the value of water saved)."""
    return cost + WATER_VALUE * (E_START - end_level)


gen, _, level_pf = window(np.arange(T), E_START, end_min=E_START)
perfect = fuel_cost(gen)
print(f"perfect foresight: {perfect:,.0f} EUR for the week, ending at {level_pf[-1]:,.0f} MWh (= the starting level)")

perfect foresight: 9,828,300 EUR for the week, ending at 3,000 MWh (= the starting level)


The **perfect-foresight** run plans the whole week at once and must leave the reservoir as full as it found it. It is the benchmark no real operator can reach.

### Day by day

Now plan one day at a time: solve the window, **commit** the first 24 hours, pass the level at the end of the committed hours to the next window, repeat. With a `lookahead`, each window is longer than the part that is committed — the operator looks beyond tomorrow but only acts on tomorrow.

In [4]:
def rolling(commit=24, lookahead=0, terminal_value=0.0):
    """Rolling horizon over the week. Returns fuel cost (EUR), level after each hour (MWh), turbine (MW)."""
    e, gens, levels, turbines = E_START, [], [], []
    for start in range(0, T, commit):
        hours = np.arange(start, min(T, start + commit + lookahead))
        g, d, lev = window(hours, e, terminal_value)
        k = min(commit, T - start)                     # commit only the first k hours
        gens.append(g.iloc[:k])
        levels.append(lev[:k])
        turbines.append(d[:k])
        e = lev[k - 1]                                 # the state handed to the next window
    return fuel_cost(pd.concat(gens, ignore_index=True)), np.concatenate(levels), np.concatenate(turbines)


cost_myopic, level_myopic, turbine_myopic = rolling()
print(f"day by day: {cost_myopic:,.0f} EUR - cheaper than perfect foresight?! - but ending at {level_myopic[-1]:,.0f} MWh")
print(f"adjusted for the water used up: {adjusted_cost(cost_myopic, level_myopic[-1]):,.0f} EUR, "
      f"{adjusted_cost(cost_myopic, level_myopic[-1]) - perfect:,.0f} EUR more than perfect foresight")

day by day: 9,575,977 EUR - cheaper than perfect foresight?! - but ending at 0 MWh
adjusted for the water used up: 9,858,577 EUR, 30,276 EUR more than perfect foresight


The day-by-day schedule looks *cheaper* than perfect foresight — 9,575,977 € against 9,828,300 € — but only because it has spent the water it was given. A one-day window sees no reason to keep any water: whatever is left at midnight is worth nothing in its objective, so it uses everything, ending every day at exactly 0 MWh. On Monday it burns almost all of the 3,000 MWh it starts with in the first five night hours, displacing the CCGT, because nothing tells it that the same water could replace coal and imports at the evening peak. The fair comparison values the water that is missing at the end of the week at its water value: with that adjustment, the myopic week costs **30,276 € more** than perfect foresight.

In [5]:
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(h, level_pf, label="perfect foresight", lw=1.5)
ax.plot(h, level_myopic, label="day by day, no value for leftover water", lw=1.2)
for d in range(1, 7):
    ax.axvline(24 * d, color="grey", lw=0.5, ls=":")
ax.set(xlabel="hour of the week", ylabel="reservoir level (MWh)", title="A one-day window empties the reservoir every day")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### PyPSA's rolling horizon — and two traps

PyPSA does the same with `n.optimize.optimize_with_rolling_horizon(horizon, overlap)`: windows of `horizon` snapshots, each starting `horizon − overlap` snapshots after the previous one, with later windows overwriting the overlapping hours. It hands the storage level on through `state_of_charge_initial`:

In [6]:
def pypsa_week():
    n = pypsa.Network()
    n.set_snapshots(h)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)
    n.add("Generator", "free", bus="HR", p_nom=free.max(), p_max_pu=free / free.max(), marginal_cost=0)
    for gen, r in thermal.iterrows():
        n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc)
    n.add("StorageUnit", "reservoir", bus="HR", p_nom=TURBINE, max_hours=E_MAX / TURBINE, p_min_pu=0.0,
          inflow=INFLOW, state_of_charge_initial=E_START, cyclic_state_of_charge=False)
    return n


pypsa_fuel = lambda n: float((n.generators_t.p[thermal.index] * thermal.mc).sum().sum())
n = pypsa_week()
n.optimize.optimize_with_rolling_horizon(horizon=24, overlap=0, solver_name="highs", log_to_console=False, progress=False)
assert np.isclose(pypsa_fuel(n), cost_myopic, rtol=1e-9)
print(f"PyPSA, day by day: {pypsa_fuel(n):,.0f} EUR - identical to the own rolling horizon")
print(f"trap 1: n.objective = {n.objective:,.0f} EUR - only the LAST window's cost, not the week's")
print(f"trap 2: state_of_charge_initial is now {n.storage_units.state_of_charge_initial['reservoir'] + 0.0:,.1f} MWh, "
      f"not {E_START:,.0f} - the run has changed the network")

PyPSA, day by day: 9,575,977 EUR - identical to the own rolling horizon
trap 1: n.objective = 1,391,714 EUR - only the LAST window's cost, not the week's
trap 2: state_of_charge_initial is now 0.0 MWh, not 3,000 - the run has changed the network


Both traps are silent. `n.objective` is overwritten by every window, so the week's cost must be computed from the dispatch. And because the function sets `state_of_charge_initial` window by window, the network no longer starts where you defined it: a second rolling run on the same network starts from the first run's last handover. **Build a fresh network for every run** (or reset `state_of_charge_initial`).

### Fix 1: look further ahead

In [7]:
rows = {"perfect foresight": {"fuel_cost": perfect, "end_level": level_pf[-1]}}
for lookahead in [0, 24, 72]:
    c, lev, _ = rolling(lookahead=lookahead)
    rows[f"commit 24 h, look ahead {lookahead} h"] = {"fuel_cost": c, "end_level": lev[-1]}
n = pypsa_week()                                          # a fresh network!
n.optimize.optimize_with_rolling_horizon(horizon=48, overlap=24, solver_name="highs", log_to_console=False, progress=False)
rows["PyPSA: horizon 48 h, overlap 24 h"] = {"fuel_cost": pypsa_fuel(n),
                                              "end_level": n.storage_units_t.state_of_charge["reservoir"].iloc[-1]}
compare = pd.DataFrame(rows).T + 0.0
compare["adjusted_cost"] = adjusted_cost(compare.fuel_cost, compare.end_level)
compare["gap_to_perfect"] = compare.adjusted_cost - perfect
compare.round(0)

                                   fuel_cost  end_level  adjusted_cost  \
perfect foresight                  9828300.0     3000.0      9828300.0   
commit 24 h, look ahead 0 h        9575977.0        0.0      9858577.0   
commit 24 h, look ahead 24 h       9568762.0        0.0      9851362.0   
commit 24 h, look ahead 72 h       9551102.0        0.0      9833702.0   
PyPSA: horizon 48 h, overlap 24 h  9569039.0        0.0      9851639.0   

                                   gap_to_perfect  
perfect foresight                             0.0  
commit 24 h, look ahead 0 h               30276.0  
commit 24 h, look ahead 24 h              23062.0  
commit 24 h, look ahead 72 h               5402.0  
PyPSA: horizon 48 h, overlap 24 h         23339.0  

Looking beyond tomorrow helps: with 24 hours of look-ahead the gap falls to 23,062 €, with 72 hours to 5,402 €. The longer window sees more of the peaks the water will be needed for, so it stops wasting it on cheap hours — but every window still ends empty, because the water left at the end of the *window* is still worth nothing to it.

PyPSA's `horizon=48, overlap=24` is the same strategy as committing 24 hours with 24 hours of look-ahead, and it lands within 277 € of the own model. The small difference is not an error: the window problems have several equally cheap solutions (lesson 3's degeneracy), and the two solvers pick different ones. Because each window hands its choice to the next, the paths drift apart a little. A rolling horizon's result is a path, not a unique optimum — compare strategies by their adjusted cost, not by their exact schedules.

### Fix 2: give leftover water its value

Looking ahead helps, but it needs a long window to see the next peak — and real forecasts get worse with distance. The alternative is to tell the short window what water is worth beyond its end: a **terminal water value**. Lesson 7 computed it for this week: 94.2 €/MWh. In PyPSA, the reward for the last hour's level is added through `extra_functionality`, which `optimize_with_rolling_horizon` passes on to every window:

In [8]:
def terminal_water_value(value):
    def add(n, snapshots):
        soc = n.model.variables["StorageUnit-state_of_charge"].sel(snapshot=snapshots[-1]).sum()
        n.model.add_objective(n.model.objective.expression - value * soc, overwrite=True)
    return add


rows = {}
for value in [WATER_VALUE, 89.1]:
    c, lev, _ = rolling(terminal_value=value)
    rows[f"own, water value {value}"] = {"fuel_cost": c, "end_level": lev[-1]}
n = pypsa_week()
n.optimize.optimize_with_rolling_horizon(horizon=24, overlap=0, solver_name="highs", log_to_console=False,
                                         progress=False, extra_functionality=terminal_water_value(WATER_VALUE))
rows[f"PyPSA, water value {WATER_VALUE}"] = {"fuel_cost": pypsa_fuel(n),
                                             "end_level": n.storage_units_t.state_of_charge["reservoir"].iloc[-1]}
valued = pd.DataFrame(rows).T + 0.0
valued["adjusted_cost"] = adjusted_cost(valued.fuel_cost, valued.end_level)
valued["gap_to_perfect"] = valued.adjusted_cost - perfect
valued.round(0)

                          fuel_cost  end_level  adjusted_cost  gap_to_perfect
own, water value 94.2     9926406.0     4041.0      9828300.0             0.0
own, water value 89.1     9572285.0        0.0      9854885.0         26585.0
PyPSA, water value 94.2  10109768.0     5988.0      9828300.0             0.0

With the right water value — lesson 7's 94.2 €/MWh — a one-day window closes the gap **completely**: the adjusted cost equals perfect foresight exactly, in the own model and in PyPSA. The reward for leftover water tells each day exactly what a MWh saved is worth to the rest of the week, which is all the information the short window was missing. The two runs end the week with different amounts of water (4,041 and 5,988 MWh) and different fuel costs, yet the same adjusted cost: at a water value equal to coal's marginal cost, storing a MWh and using it to replace coal are worth exactly the same, so every split between them is optimal.

A water value on the wrong side of a threshold is almost as bad as none: at 89.1 €/MWh — the CCGT's cost — the reservoir still empties every day and the gap is 26,585 €. (The exercise maps how wide the band of good values is.) That is the practical lesson: real operators run exactly this two-level scheme — a long-term model (months to a year of inflows and prices) estimates water values, and the short-term model uses them as the value of the water it leaves behind. The quality of the short-term schedule depends on how good those water values are.

## Check your understanding

**The end-of-horizon problem**

1. Why does a one-day window empty the reservoir, even though water is valuable tomorrow? What exactly is missing from its objective?
2. Day by day, the fuel cost was *lower* than with perfect foresight. Why is that not a better result — and how does the adjusted cost settle it?

**Water values in practice**

1. The right water value closed the gap completely; one on the wrong side of the CCGT's cost did not. Where would a real operator get the water value from, and what makes it uncertain?
2. Lesson 6 needed the units' up and down times before the day. What else besides the reservoir level must be handed from one window to the next in a real rolling-horizon model?